# 1. 라이브러리 로드 + 데이터 불러오기

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import ast
from collections import Counter

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

DATA_PATH = Path("results/result_sample_shorts_all_for_video_agent_fixed.csv")

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")

print("데이터 크기:", df.shape)

데이터 크기: (200, 75)


# 2. 컬럼 확인

In [2]:
for i, col in enumerate(df.columns, start=1):
    print(i, col)

1 video_id
2 production_quality
3 lighting_style
4 color_mood
5 editing_pace
6 motion_graphic
7 video_format
8 first_3sec
9 background_style
10 top_colors
11 person_ratio
12 face_ratio
13 text_ratio
14 reason
15 avg_brightness
16 avg_blue
17 avg_green
18 avg_red
19 채널명
20 domain
21 success_label
22 source_checkpoint
23 title
24 channel_id
25 description
26 업로드일시
27 tags
28 조회수
29 좋아요수
30 댓글수
31 영상길이(초)
32 definition
33 license
34 embeddable
35 has_paid_product_placement
36 thumbnail
37 caption
38 final_url
39 instream_type
40 channel_handle
41 channel_tier
42 구독자수
43 description_missing_flag
44 tags_missing_flag
45 참여율(ER)
46 조회수 대비 좋아요율
47 조회수 대비 댓글률
48 wei
49 description_length
50 category_name
51 upload_year
52 upload_month
53 upload_dayofweek
54 upload_hour
55 tags_count
56 upload_quarter
57 upload_ym_quarter
58 upload_ymd
59 경과일수
60 도달률(RR)
61 일평균 조회수
62 RR_백분위
63 ER_백분위
64 score1
65 조회수성과
66 좋아요성과
67 댓글성과
68 조회수성과_상위1%
69 조회수성과_상위5%
70 좋아요성과_상위1%
71 좋아요성과_상위5%
72 댓글성과_상위1%
73 댓글성

# 3. 기본 품질 체크

In [3]:
print("전체 행 수:", len(df))
print("중복 video_id 수:", df["video_id"].duplicated().sum())

display(
    df.groupby(["domain", "success_label"])
      .size()
      .reset_index(name="count")
)

전체 행 수: 200
중복 video_id 수: 0


,domain,success_label,count
0,FnB,fail,52
1,FnB,success,48
2,IT,fail,58
3,IT,success,42


In [4]:
missing_summary = (
    df.isna()
      .sum()
      .reset_index()
)

missing_summary.columns = ["column", "missing_count"]
missing_summary["missing_ratio"] = (missing_summary["missing_count"] / len(df)).round(3)

missing_summary = missing_summary.sort_values("missing_count", ascending=False)

display(missing_summary[missing_summary["missing_count"] > 0])

,column,missing_count,missing_ratio
66,댓글성과,14,0.07


In [5]:
ratio_cols = ["person_ratio", "face_ratio", "text_ratio"]

for col in ratio_cols:
    print(col)
    print("min:", df[col].min())
    print("max:", df[col].max())
    print("mean:", round(df[col].mean(), 3))
    print()

person_ratio
min: 0.0
max: 1.0
mean: 0.689

face_ratio
min: 0.0
max: 1.0
mean: 0.555

text_ratio
min: 0.0
max: 1.0
mean: 0.733



# 4. 분석에 사용할 컬럼 정의

In [6]:
numeric_cols = [
    "avg_brightness",
    "avg_blue",
    "avg_green",
    "avg_red",
    "person_ratio",
    "face_ratio",
    "text_ratio",
]

numeric_cols = [col for col in numeric_cols if col in df.columns]

categorical_cols = [
    "production_quality",
    "lighting_style",
    "color_mood",
    "editing_pace",
    "motion_graphic",
    "video_format",
    "first_3sec",
    "background_style",
]

categorical_cols = [col for col in categorical_cols if col in df.columns]

print("수치형 컬럼:", numeric_cols)
print("범주형 컬럼:", categorical_cols)

수치형 컬럼: ['avg_brightness', 'avg_blue', 'avg_green', 'avg_red', 'person_ratio', 'face_ratio', 'text_ratio']
범주형 컬럼: ['production_quality', 'lighting_style', 'color_mood', 'editing_pace', 'motion_graphic', 'video_format', 'first_3sec', 'background_style']


# 5. 전체 수치형 기초 통계

In [7]:
display(df[numeric_cols].describe().T.round(3))

,count,mean,std,min,25%,50%,75%,max
avg_brightness,200.0,109.211,46.143,26.34,71.818,113.145,136.522,231.97
avg_blue,200.0,106.688,49.538,17.65,69.188,107.900,134.685,224.30
avg_green,200.0,106.673,47.519,20.05,68.330,110.800,137.282,229.96
avg_red,200.0,115.496,49.291,25.27,75.180,115.875,148.715,239.64
person_ratio,200.0,0.689,0.324,0.00,0.500,0.840,0.950,1.00
face_ratio,200.0,0.555,0.351,0.00,0.165,0.700,0.850,1.00
text_ratio,200.0,0.733,0.208,0.00,0.600,0.800,0.900,1.00


# 6. 도메인 × 성공/실패별 수치형 평균

In [8]:
numeric_mean_summary = (
    df.groupby(["domain", "success_label"])[numeric_cols]
      .mean()
      .round(3)
      .reset_index()
)

display(numeric_mean_summary)

,domain,success_label,avg_brightness,avg_blue,avg_green,avg_red,person_ratio,face_ratio,text_ratio
0,FnB,fail,124.562,111.943,120.713,137.047,0.602,0.399,0.652
1,FnB,success,103.751,93.767,99.241,115.789,0.786,0.649,0.712
2,IT,fail,97.767,103.477,97.262,97.780,0.734,0.641,0.769
3,IT,success,112.247,119.381,110.782,112.947,0.621,0.524,0.807


In [ ]:
# 중앙값
numeric_median_summary = (
    df.groupby(["domain", "success_label"])[numeric_cols]
      .median()
      .round(3)
      .reset_index()
)

display(numeric_median_summary)

,domain,success_label,avg_brightness,avg_blue,avg_green,avg_red,person_ratio,face_ratio,text_ratio
0,FnB,fail,133.335,109.280,124.08,147.075,0.75,0.450,0.700
1,FnB,success,96.900,87.635,95.37,109.060,0.90,0.775,0.725
2,IT,fail,86.770,93.720,87.70,86.660,0.85,0.740,0.800
3,IT,success,115.150,118.355,112.80,116.005,0.80,0.700,0.800


# 7. 성공 - 실패 수치 차이 계산

In [10]:
diff_rows = []

for domain in df["domain"].unique():
    temp = df[df["domain"] == domain]
    
    success_mean = temp[temp["success_label"] == "success"][numeric_cols].mean()
    fail_mean = temp[temp["success_label"] == "fail"][numeric_cols].mean()
    
    for col in numeric_cols:
        diff_rows.append({
            "domain": domain,
            "feature": col,
            "success_mean": round(success_mean[col], 3),
            "fail_mean": round(fail_mean[col], 3),
            "diff_success_minus_fail": round(success_mean[col] - fail_mean[col], 3)
        })

numeric_diff_df = pd.DataFrame(diff_rows)

display(
    numeric_diff_df.sort_values(
        ["domain", "diff_success_minus_fail"],
        ascending=[True, False]
    )
)

,domain,feature,success_mean,fail_mean,diff_success_minus_fail
5,FnB,face_ratio,0.649,0.399,0.250
4,FnB,person_ratio,0.786,0.602,0.185
6,FnB,text_ratio,0.712,0.652,0.060
1,FnB,avg_blue,93.767,111.943,-18.176
0,FnB,avg_brightness,103.751,124.562,-20.811
3,FnB,avg_red,115.789,137.047,-21.258
2,FnB,avg_green,99.241,120.713,-21.471
8,IT,avg_blue,119.381,103.477,15.904
10,IT,avg_red,112.947,97.780,15.167
7,IT,avg_brightness,112.247,97.767,14.480


# 8. 범주형 변수 전체 빈도 확인

In [11]:
for col in categorical_cols:
    print("=" * 80)
    print(col)
    display(
        df[col]
        .value_counts(dropna=False)
        .reset_index()
        .rename(columns={"index": col, col: "count"})
    )

production_quality


,count,count
0,프로페셔널,109
1,고퀄리티,70
2,일반,21


lighting_style


,count,count
0,인공조명,151
1,혼합,31
2,자연광,18


color_mood


...

,count,count
0,광고/CF,81
1,기술설명,22
2,웹예능,18
3,인터뷰,18
4,튜토리얼,15
5,이벤트/행사,13
6,제품리뷰,8
7,에피소드소개,6
8,브이로그,5
9,웹드라마,5


first_3sec


,count,count
0,텍스트,111
1,인물,60
2,제품,19
3,장면,6
4,음식,3
5,기업 로고,1


background_style


,count,count
0,실내,99
1,혼합,45
2,스튜디오,42
3,실외,14


# 9. 도메인 × 성공/실패별 범주 비율표

In [12]:
def make_category_ratio_table(df, category_col, group_cols=["domain", "success_label"]):
    count_df = (
        df.groupby(group_cols + [category_col])
          .size()
          .reset_index(name="count")
    )
    
    total_df = (
        df.groupby(group_cols)
          .size()
          .reset_index(name="total")
    )
    
    result = count_df.merge(total_df, on=group_cols, how="left")
    result["ratio"] = (result["count"] / result["total"]).round(3)
    
    return result.sort_values(group_cols + ["ratio"], ascending=[True, True, False])


for col in categorical_cols:
    print("=" * 80)
    print(col)
    display(make_category_ratio_table(df, col))

production_quality


,domain,success_label,production_quality,count,total,ratio
2,FnB,fail,프로페셔널,34,52,0.654
0,FnB,fail,고퀄리티,15,52,0.288
1,FnB,fail,일반,3,52,0.058
5,FnB,success,프로페셔널,27,48,0.562
3,FnB,success,고퀄리티,15,48,0.312
4,FnB,success,일반,6,48,0.125
8,IT,fail,프로페셔널,30,58,0.517
6,IT,fail,고퀄리티,24,58,0.414
7,IT,fail,일반,4,58,0.069
11,IT,success,프로페셔널,18,42,0.429


lighting_style


,domain,success_label,lighting_style,count,total,ratio
0,FnB,fail,인공조명,40,52,0.769
2,FnB,fail,혼합,7,52,0.135
1,FnB,fail,자연광,5,52,0.096
3,FnB,success,인공조명,34,48,0.708
5,FnB,success,혼합,10,48,0.208
4,FnB,success,자연광,4,48,0.083
6,IT,fail,인공조명,47,58,0.810
8,IT,fail,혼합,7,58,0.121
7,IT,fail,자연광,4,58,0.069
9,IT,success,인공조명,30,42,0.714


color_mood


...

,domain,success_label,video_format,count,total,ratio
0,FnB,fail,광고/CF,30,52,0.577
10,FnB,fail,튜토리얼,8,52,0.154
7,FnB,fail,이벤트/행사,4,52,0.077
3,FnB,fail,애니메이션,2,52,0.038
8,FnB,fail,인터뷰,2,52,0.038
1,FnB,fail,브이로그,1,52,0.019
2,FnB,fail,시설소개,1,52,0.019
4,FnB,fail,에피소드소개,1,52,0.019
5,FnB,fail,요리/레시피,1,52,0.019
6,FnB,fail,웹예능,1,52,0.019


first_3sec


,domain,success_label,first_3sec,count,total,ratio
5,FnB,fail,텍스트,28,52,0.538
2,FnB,fail,인물,11,52,0.212
4,FnB,fail,제품,9,52,0.173
1,FnB,fail,음식,2,52,0.038
0,FnB,fail,기업 로고,1,52,0.019
3,FnB,fail,장면,1,52,0.019
7,FnB,success,인물,25,48,0.521
10,FnB,success,텍스트,12,48,0.250
9,FnB,success,제품,8,48,0.167
8,FnB,success,장면,2,48,0.042


background_style


,domain,success_label,background_style,count,total,ratio
1,FnB,fail,실내,25,52,0.481
0,FnB,fail,스튜디오,12,52,0.231
3,FnB,fail,혼합,10,52,0.192
2,FnB,fail,실외,5,52,0.096
5,FnB,success,실내,21,48,0.438
7,FnB,success,혼합,13,48,0.271
4,FnB,success,스튜디오,12,48,0.250
6,FnB,success,실외,2,48,0.042
9,IT,fail,실내,33,58,0.569
8,IT,fail,스튜디오,12,58,0.207


# 10. 성공/실패 차이가 큰 범주 찾기

In [13]:
def make_success_fail_category_diff(df, category_col):
    ratio_df = make_category_ratio_table(df, category_col)
    
    pivot = ratio_df.pivot_table(
        index=["domain", category_col],
        columns="success_label",
        values="ratio",
        fill_value=0
    ).reset_index()
    
    if "success" not in pivot.columns:
        pivot["success"] = 0
    if "fail" not in pivot.columns:
        pivot["fail"] = 0
    
    pivot["diff_success_minus_fail"] = (pivot["success"] - pivot["fail"]).round(3)
    
    return pivot.sort_values(
        ["domain", "diff_success_minus_fail"],
        ascending=[True, False]
    )


category_diff_dict = {}

for col in categorical_cols:
    print("=" * 80)
    print(col)
    diff_table = make_success_fail_category_diff(df, col)
    category_diff_dict[col] = diff_table
    display(diff_table)

production_quality


success_label,domain,production_quality,fail,success,diff_success_minus_fail
1,FnB,일반,0.058,0.125,0.067
0,FnB,고퀄리티,0.288,0.312,0.024
2,FnB,프로페셔널,0.654,0.562,-0.092
4,IT,일반,0.069,0.190,0.121
3,IT,고퀄리티,0.414,0.381,-0.033
5,IT,프로페셔널,0.517,0.429,-0.088


lighting_style


success_label,domain,lighting_style,fail,success,diff_success_minus_fail
2,FnB,혼합,0.135,0.208,0.073
1,FnB,자연광,0.096,0.083,-0.013
0,FnB,인공조명,0.769,0.708,-0.061
4,IT,자연광,0.069,0.119,0.050
5,IT,혼합,0.121,0.167,0.046
3,IT,인공조명,0.810,0.714,-0.096


color_mood


...

success_label,domain,video_format,fail,success,diff_success_minus_fail
7,FnB,웹예능,0.019,0.167,0.148
10,FnB,제품리뷰,0.019,0.083,0.064
6,FnB,웹드라마,0.000,0.062,0.062
8,FnB,이벤트/행사,0.077,0.104,0.027
2,FnB,시설소개,0.019,0.021,0.002
4,FnB,에피소드소개,0.019,0.021,0.002
3,FnB,애니메이션,0.038,0.021,-0.017
9,FnB,인터뷰,0.038,0.021,-0.017
1,FnB,브이로그,0.019,0.000,-0.019
5,FnB,요리/레시피,0.019,0.000,-0.019


first_3sec


success_label,domain,first_3sec,fail,success,diff_success_minus_fail
2,FnB,인물,0.212,0.521,0.309
3,FnB,장면,0.019,0.042,0.023
4,FnB,제품,0.173,0.167,-0.006
1,FnB,음식,0.038,0.021,-0.017
0,FnB,기업 로고,0.019,0.000,-0.019
5,FnB,텍스트,0.538,0.250,-0.288
9,IT,텍스트,0.621,0.833,0.212
8,IT,제품,0.034,0.000,-0.034
7,IT,장면,0.052,0.000,-0.052
6,IT,인물,0.293,0.167,-0.126


background_style


success_label,domain,background_style,fail,success,diff_success_minus_fail
3,FnB,혼합,0.192,0.271,0.079
0,FnB,스튜디오,0.231,0.250,0.019
1,FnB,실내,0.481,0.438,-0.043
2,FnB,실외,0.096,0.042,-0.054
7,IT,혼합,0.155,0.310,0.155
6,IT,실외,0.069,0.071,0.002
4,IT,스튜디오,0.207,0.143,-0.064
5,IT,실내,0.569,0.476,-0.093


# 11. 핵심 변수만 따로 보기

In [14]:
key_categorical_cols = [
    "first_3sec",
    "motion_graphic",
    "editing_pace",
    "video_format",
]

for col in key_categorical_cols:
    if col in df.columns:
        print("=" * 80)
        print(col)
        display(make_success_fail_category_diff(df, col))

first_3sec


success_label,domain,first_3sec,fail,success,diff_success_minus_fail
2,FnB,인물,0.212,0.521,0.309
3,FnB,장면,0.019,0.042,0.023
4,FnB,제품,0.173,0.167,-0.006
1,FnB,음식,0.038,0.021,-0.017
0,FnB,기업 로고,0.019,0.000,-0.019
5,FnB,텍스트,0.538,0.250,-0.288
9,IT,텍스트,0.621,0.833,0.212
8,IT,제품,0.034,0.000,-0.034
7,IT,장면,0.052,0.000,-0.052
6,IT,인물,0.293,0.167,-0.126


motion_graphic


success_label,domain,motion_graphic,fail,success,diff_success_minus_fail
0,FnB,보조적,0.538,0.854,0.316
1,FnB,없음,0.019,0.021,0.002
2,FnB,핵심요소,0.442,0.125,-0.317
4,IT,핵심요소,0.276,0.548,0.272
3,IT,보조적,0.724,0.452,-0.272


editing_pace


success_label,domain,editing_pace,fail,success,diff_success_minus_fail
3,FnB,빠름,0.462,0.625,0.163
0,FnB,매우 느림,0.038,0.021,-0.017
2,FnB,보통,0.115,0.083,-0.032
1,FnB,매우 빠름,0.385,0.271,-0.114
6,IT,매우 빠름,0.172,0.262,0.090
8,IT,빠름,0.466,0.548,0.082
4,IT,느림,0.017,0.000,-0.017
5,IT,매우 느림,0.069,0.000,-0.069
7,IT,보통,0.276,0.190,-0.086


video_format


success_label,domain,video_format,fail,success,diff_success_minus_fail
7,FnB,웹예능,0.019,0.167,0.148
10,FnB,제품리뷰,0.019,0.083,0.064
6,FnB,웹드라마,0.000,0.062,0.062
8,FnB,이벤트/행사,0.077,0.104,0.027
2,FnB,시설소개,0.019,0.021,0.002
4,FnB,에피소드소개,0.019,0.021,0.002
3,FnB,애니메이션,0.038,0.021,-0.017
9,FnB,인터뷰,0.038,0.021,-0.017
1,FnB,브이로그,0.019,0.000,-0.019
5,FnB,요리/레시피,0.019,0.000,-0.019


# 12. top_colors 분석12. top_colors 분석

In [ ]:
def parse_top_colors(x):
    if pd.isna(x):
        return []
    
    if isinstance(x, list):
        return x
    
    try:
        parsed = ast.literal_eval(x)
        if isinstance(parsed, list):
            return parsed
        return [str(parsed)]
    except:
        return [str(x)]

df["top_colors_parsed"] = df["top_colors"].apply(parse_top_colors)

display(df[["top_colors", "top_colors_parsed"]].head())

# top_colors_parsed 컬럼은 

,top_colors,top_colors_parsed
0,"['노란색', '흰색', '빨간색']","[노란색, 흰색, 빨간색]"
1,"['회색', '녹색', '검은색']","[회색, 녹색, 검은색]"
2,"['빨간색', '흰색', '파란색']","[빨간색, 흰색, 파란색]"
3,"['빨간색', '검은색', '흰색']","[빨간색, 검은색, 흰색]"
4,"['갈색', '금색', '흰색']","[갈색, 금색, 흰색]"


In [16]:
def color_counter_by_group(df, group_cols=["domain", "success_label"], top_n=10):
    rows = []
    
    for group_key, group_df in df.groupby(group_cols):
        colors = []
        
        for color_list in group_df["top_colors_parsed"]:
            colors.extend(color_list)
        
        counter = Counter(colors)
        total = sum(counter.values())
        
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        
        for color, count in counter.most_common(top_n):
            row = dict(zip(group_cols, group_key))
            row.update({
                "color": color,
                "count": count,
                "ratio": round(count / total, 3) if total else 0
            })
            rows.append(row)
    
    return pd.DataFrame(rows)

color_summary = color_counter_by_group(df)
display(color_summary)

,domain,success_label,color,count,ratio
0,FnB,fail,흰색,41,0.263
1,FnB,fail,빨간색,17,0.109
2,FnB,fail,갈색,17,0.109
3,FnB,fail,노란색,16,0.103
4,FnB,fail,파란색,12,0.077
5,FnB,fail,검은색,11,0.071
6,FnB,fail,주황색,8,0.051
7,FnB,fail,회색,7,0.045
8,FnB,fail,초록색,6,0.038
9,FnB,fail,분홍색,3,0.019


# 13. 인사이트 후보 자동 정리용 표 만들기

In [17]:
numeric_insight_candidates = numeric_diff_df.copy()
numeric_insight_candidates["abs_diff"] = numeric_insight_candidates["diff_success_minus_fail"].abs()

display(
    numeric_insight_candidates
    .sort_values(["domain", "abs_diff"], ascending=[True, False])
)

,domain,feature,success_mean,fail_mean,diff_success_minus_fail,abs_diff
2,FnB,avg_green,99.241,120.713,-21.471,21.471
3,FnB,avg_red,115.789,137.047,-21.258,21.258
0,FnB,avg_brightness,103.751,124.562,-20.811,20.811
1,FnB,avg_blue,93.767,111.943,-18.176,18.176
5,FnB,face_ratio,0.649,0.399,0.250,0.250
4,FnB,person_ratio,0.786,0.602,0.185,0.185
6,FnB,text_ratio,0.712,0.652,0.060,0.060
8,IT,avg_blue,119.381,103.477,15.904,15.904
10,IT,avg_red,112.947,97.780,15.167,15.167
7,IT,avg_brightness,112.247,97.767,14.480,14.480


In [ ]:
category_insight_rows = []

for col, table in category_diff_dict.items():
    temp = table.copy()
    temp["feature"] = col
    temp["category"] = temp[col]
    temp["abs_diff"] = temp["diff_success_minus_fail"].abs()
    
    category_insight_rows.append(
        temp[["domain", "feature", "category", "success", "fail", "diff_success_minus_fail", "abs_diff"]]
    )

category_insight_candidates = pd.concat(category_insight_rows, ignore_index=True)

display(
    category_insight_candidates
    .sort_values(["domain", "abs_diff"], ascending=[True, False])
    .head(50)
)
# 

success_label,domain,feature,category,success,fail,diff_success_minus_fail,abs_diff
32,FnB,motion_graphic,핵심요소,0.125,0.442,-0.317,0.317
30,FnB,motion_graphic,보조적,0.854,0.538,0.316,0.316
60,FnB,first_3sec,인물,0.521,0.212,0.309,0.309
65,FnB,first_3sec,텍스트,0.250,0.538,-0.288,0.288
21,FnB,editing_pace,빠름,0.625,0.462,0.163,0.163
35,FnB,video_format,웹예능,0.167,0.019,0.148,0.148
46,FnB,video_format,광고/CF,0.438,0.577,-0.139,0.139
12,FnB,color_mood,중립,0.208,0.077,0.131,0.131
24,FnB,editing_pace,매우 빠름,0.271,0.385,-0.114,0.114
15,FnB,color_mood,비비드,0.479,0.577,-0.098,0.098


# 14. 결과 저장

In [19]:
# OUTPUT_DIR = Path("eda_outputs")
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# numeric_mean_summary.to_csv(
#     OUTPUT_DIR / "shorts_numeric_mean_summary.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# numeric_diff_df.to_csv(
#     OUTPUT_DIR / "shorts_numeric_success_fail_diff.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# category_insight_candidates.to_csv(
#     OUTPUT_DIR / "shorts_category_insight_candidates.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# color_summary.to_csv(
#     OUTPUT_DIR / "shorts_top_colors_summary.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# print("저장 완료")